### Chapter 9: Evaluate the Performance of Machine Learning Algorithms with Resampling

##### 9.1 Evaluate Machine Learning Algorithms
Why can’t you train your machine learning algorithm on your dataset and use predictions from
this same dataset to evaluate machine learning algorithms? The simple answer is overfitting.
Imagine an algorithm that remembers every observation it is shown during training. If you
evaluated your machine learning algorithm on the same dataset used to train the algorithm, then
an algorithm like this would have a perfect score on the training dataset. But the predictions it
made on new data would be terrible. We must evaluate our machine learning algorithms on
data that is not used to train the algorithm.
The evaluation is an estimate that we can use to talk about how well we think the algorithm
may actually do in practice. It is not a guarantee of performance. Once we estimate the
performance of our algorithm, we can then re-train the final algorithm on the entire training
dataset and get it ready for operational use. Next up we are going to look at four different
techniques that we can use to split up our training dataset and create useful estimates of
performance for our machine learning algorithms:
 + Train and Test Sets.
 + k-fold Cross Validation.
 + Leave One Out Cross Validation.
 + Repeated Random Test-Train Splits.

##### 9.2 Split into Train and Test Sets
The simplest method that we can use to evaluate the performance of a machine learning
algorithm is to use different training and testing datasets. We can take our original dataset and
split it into two parts. Train the algorithm on the first part, make predictions on the second
part and evaluate the predictions against the expected results. The size of the split can depend
on the size and specifics of your dataset, although it is common to use 67% of the data for
training and the remaining 33% for testing.
This algorithm evaluation technique is very fast. It is ideal for large datasets (millions of
records) where there is strong evidence that both splits of the data are representative of the
underlying problem. Because of the speed, it is useful to use this approach when the algorithm
you are investigating is slow to train. A downside of this technique is that it can have a high
variance. This means that differences in the training and test dataset can result in meaningful
differences in the estimate of accuracy. In the example below we split the Pima Indians dataset
into 67%/33% splits for training and test and evaluate the accuracy of a Logistic Regression
model.

In [ ]:
# Evaluate using a train and a test set
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
test_size = 0.33
seed = 7
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=test_size,
random_state=seed)
model = LogisticRegression()
model.fit(X_train, Y_train)
result = model.score(X_test, Y_test)
print("Accuracy: %.3f%%") % (result*100.0)

We can see that the estimated accuracy for the model was approximately 75%. Note that
in addition to specifying the size of the split, we also specify the random seed. Because the
split of the data is random, we want to ensure that the results are reproducible. By specifying
the random seed we ensure that we get the same random numbers each time we run the code
and in turn the same split of data. This is important if we want to compare this result to
the estimated accuracy of another machine learning algorithm or the same algorithm with a
different configuration. To ensure the comparison was apples-for-apples, we must ensure that
they are trained and tested on exactly the same data.

Accuracy: 75.591%

##### 9.3 K-fold Cross Validation
Cross validation is an approach that you can use to estimate the performance of a machine
learning algorithm with less variance than a single train-test set split. It works by splitting
the dataset into k-parts (e.g. k = 5 or k = 10). Each split of the data is called a fold. The
algorithm is trained on k − 1 folds with one held back and tested on the held back fold. This is
repeated so that each fold of the dataset is given a chance to be the held back test set. After
running cross validation you end up with k different performance scores that you can summarize
using a mean and a standard deviation.
The result is a more reliable estimate of the performance of the algorithm on new data. It is
more accurate because the algorithm is trained and evaluated multiple times on different data.
The choice of k must allow the size of each test partition to be large enough to be a reasonable
sample of the problem, whilst allowing enough repetitions of the train-test evaluation of the
algorithm to provide a fair estimate of the algorithms performance on unseen data. For modest
sized datasets in the thousands or tens of thousands of records, k values of 3, 5 and 10 are
common. In the example below we use 10-fold cross validation.

In [ ]:
# Evaluate using Cross Validation
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
num_folds = 10
seed = 7
kfold = KFold(n_splits=num_folds, random_state=seed)
model = LogisticRegression()
results = cross_val_score(model, X, Y, cv=kfold)
print("Accuracy: %.3f%% (%.3f%%)") % (results.mean()*100.0, results.std()*100.0)

You can see that we report both the mean and the standard deviation of the performance
measure. When summarizing performance measures, it is a good practice to summarize the
distribution of the measures, in this case assuming a Gaussian distribution of performance (a
very reasonable assumption) and recording the mean and standard deviation.

Accuracy: 76.951% (4.841%)

##### 9.4 Leave One Out Cross Validation
You can configure cross validation so that the size of the fold is 1 (k is set to the number of
observations in your dataset). This variation of cross validation is called leave-one-out cross
validation. The result is a large number of performance measures that can be summarized in an effort to give a more reasonable estimate of the accuracy of your model on unseen data.
A downside is that it can be a computationally more expensive procedure than k-fold cross
validation. In the example below we use leave-one-out cross validation.

In [ ]:
# Evaluate using Leave One Out Cross Validation
from pandas import read_csv
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
num_folds = 10
loocv = LeaveOneOut()
model = LogisticRegression()
results = cross_val_score(model, X, Y, cv=loocv)
print("Accuracy: %.3f%% (%.3f%%)") % (results.mean()*100.0, results.std()*100.0)

You can see in the standard deviation that the score has more variance than the k-fold cross
validation results described above.

Accuracy: 76.823% (42.196%)

##### 9.5 Repeated Random Test-Train Splits
Another variation on k-fold cross validation is to create a random split of the data like the
train/test split described above, but repeat the process of splitting and evaluation of the
algorithm multiple times, like cross validation. This has the speed of using a train/test split and
the reduction in variance in the estimated performance of k-fold cross validation. You can also
repeat the process many more times as needed to improve the accuracy. A down side is that
repetitions may include much of the same data in the train or the test split from run to run,
introducing redundancy into the evaluation. The example below splits the data into a 67%/33%
train/test split and repeats the process 10 times.

In [ ]:
# Evaluate using Shuffle Split Cross Validation
from pandas import read_csv
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
n_splits = 10
test_size = 0.33
seed = 7
kfold = ShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=seed)
model = LogisticRegression()
results = cross_val_score(model, X, Y, cv=kfold)
print("Accuracy: %.3f%% (%.3f%%)") % (results.mean()*100.0, results.std()*100.0)

We can see that in this case the distribution of the performance measure is on par with
k-fold cross validation above.

Accuracy: 76.496% (1.698%)

##### 9.6 What Techniques to Use When

This section lists some tips to consider what resampling technique to use in different circum-
stances.

 + Generally k-fold cross validation is the gold standard for evaluating the performance of a
machine learning algorithm on unseen data with k set to 3, 5, or 10.
 + Using a train/test split is good for speed when using a slow algorithm and produces
performance estimates with lower bias when using large datasets.
 + Techniques like leave-one-out cross validation and repeated random splits can be useful
intermediates when trying to balance variance in the estimated performance, model
training speed and dataset size.
The best advice is to experiment and find a technique for your problem that is fast and
produces reasonable estimates of performance that you can use to make decisions. If in doubt,
use 10-fold cross validation.

##### 9.7 Summary
In this chapter you discovered statistical techniques that you can use to estimate the performance
of your machine learning algorithms, called resampling. Specifically, you learned about:
 + Train and Test Sets.
 + Cross Validation.
 + Leave One Out Cross Validation.
 + Repeated Random Test-Train Splits.


9.7.1 Next
In the next section you will learn how you can evaluate the performance of classification and
regression algorithms using a suite of different metrics and built in evaluation reports.

### Chapter 10: Machine Learning Algorithm Performance Metrics

The metrics that you choose to evaluate your machine learning algorithms are very important.
Choice of metrics influences how the performance of machine learning algorithms is measured
and compared. They influence how you weight the importance of different characteristics in
the results and your ultimate choice of which algorithm to choose. In this chapter you will
discover how to select and use different machine learning performance metrics in Python with
scikit-learn. Let’s get started.

##### 10.1 Algorithm Evaluation Metrics

In this lesson, various different algorithm evaluation metrics are demonstrated for both classifi-
cation and regression type machine learning problems. In each recipe, the dataset is downloaded

directly from the UCI Machine Learning repository.

+ For classification metrics, the Pima Indians onset of diabetes dataset is used as demon-
stration. This is a binary classification problem where all of the input variables are

numeric.
+ For regression metrics, the Boston House Price dataset is used as demonstration. this is a
regression problem where all of the input variables are also numeric.
All recipes evaluate the same algorithms, Logistic Regression for classification and Linear
Regression for the regression problems. A 10-fold cross validation test harness is used to
demonstrate each metric, because this is the most likely scenario you will use when employing
different algorithm evaluation metrics.
A caveat in these recipes is the cross validation.cross val score function1 used to
report the performance in each recipe. It does allow the use of different scoring metrics
that will be discussed, but all scores are reported so that they can be sorted in ascending
order (largest score is best). Some evaluation metrics (like mean squared error) are naturally
descending scores (the smallest score is best) and as such are reported as negative by the 
cross validation.cross val score() function. This is important to note, because some
scores will be reported as negative that by definition can never be negative. I will remind you
about this caveat as we work through the lesson.
You can learn more about machine learning algorithm performance metrics supported by
scikit-learn on the page Model evaluation: quantifying the quality of predictions2. Let’s get on
with the evaluation metrics.

##### 10.2 Classification Metrics
Classification problems are perhaps the most common type of machine learning problem and as
such there are a myriad of metrics that can be used to evaluate predictions for these problems.
In this section we will review how to use the following metrics:
 + Classification Accuracy.
 + Logarithmic Loss.
 + Area Under ROC Curve.
 + Confusion Matrix.
 + Classification Report.

##### 10.2.1 Classification Accuracy
Classification accuracy is the number of correct predictions made as a ratio of all predictions
made. This is the most common evaluation metric for classification problems, it is also the most
misused. It is really only suitable when there are an equal number of observations in each class
(which is rarely the case) and that all predictions and prediction errors are equally important,
which is often not the case. Below is an example of calculating classification accuracy.

In [ ]:
# Cross Validation Classification Accuracy
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
kfold = KFold(n_splits=10, random_state=7)
model = LogisticRegression()
scoring = 'accuracy'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print("Accuracy: %.3f (%.3f)") % (results.mean(), results.std())

You can see that the ratio is reported. This can be converted into a percentage by multiplying
the value by 100, giving an accuracy score of approximately 77% accurate.

Accuracy: 0.770 (0.048)

##### 10.2.2 Logarithmic Loss
Logarithmic loss (or logloss) is a performance metric for evaluating the predictions of probabilities
of membership to a given class. The scalar probability between 0 and 1 can be seen as a measure
of confidence for a prediction by an algorithm. Predictions that are correct or incorrect are
rewarded or punished proportionally to the confidence of the prediction. Below is an example
of calculating logloss for Logistic regression predictions on the Pima Indians onset of diabetes
dataset.

In [ ]:
# Cross Validation Classification LogLoss
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
kfold = KFold(n_splits=10, random_state=7)
model = LogisticRegression()
scoring = 'neg_log_loss'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print("Logloss: %.3f (%.3f)") % (results.mean(), results.std())

Smaller logloss is better with 0 representing a perfect logloss. As mentioned above, the
measure is inverted to be ascending when using the cross val score() function.

Logloss: -0.493 (0.047)

##### 10.2.3 Area Under ROC Curve
Area under ROC Curve (or AUC for short) is a performance metric for binary classification
problems. The AUC represents a model’s ability to discriminate between positive and negative
classes. An area of 1.0 represents a model that made all predictions perfectly. An area of
0.5 represents a model that is as good as random. ROC can be broken down into sensitivity
and specificity. A binary classification problem is really a trade-off between sensitivity and
specificity.
 + Sensitivity is the true positive rate also called the recall. It is the number of instances
from the positive (first) class that actually predicted correctly.

 + Specificity is also called the true negative rate. Is the number of instances from the
negative (second) class that were actually predicted correctly.

In [ ]:
# Cross Validation Classification ROC AUC
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
kfold = KFold(n_splits=10, random_state=7)
model = LogisticRegression()
scoring = 'roc_auc'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print("AUC: %.3f (%.3f)") % (results.mean(), results.std())

You can see the AUC is relatively close to 1 and greater than 0.5, suggesting some skill in
the predictions

AUC: 0.824 (0.041)

##### 10.2.4 Confusion Matrix
The confusion matrix is a handy presentation of the accuracy of a model with two or more
classes. The table presents predictions on the x-axis and accuracy outcomes on the y-axis. The
cells of the table are the number of predictions made by a machine learning algorithm. For
example, a machine learning algorithm can predict 0 or 1 and each prediction may actually have
been a 0 or 1. Predictions for 0 that were actually 0 appear in the cell for prediction = 0 and
actual = 0, whereas predictions for 0 that were actually 1 appear in the cell for prediction = 0
and actual = 1. And so on. Below is an example of calculating a confusion matrix for a set of
predictions by a Logistic Regression on the Pima Indians onset of diabetes dataset.

In [ ]:
# Cross Validation Classification Confusion Matrix
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
test_size = 0.33
seed = 7
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=test_size,
random_state=seed)
model = LogisticRegression()
model.fit(X_train, Y_train)
predicted = model.predict(X_test)
matrix = confusion_matrix(Y_test, predicted)
print(matrix)

Although the array is printed without headings, you can see that the majority of the
predictions fall on the diagonal line of the matrix (which are correct predictions).

[[141 21]
[ 41 51]]

##### 10.2.5 Classification Report

The scikit-learn library provides a convenience report when working on classification prob-
lems to give you a quick idea of the accuracy of a model using a number of measures. The

classification report() function displays the precision, recall, F1-score and support for each
class. The example below demonstrates the report on the binary classification problem.

In [ ]:
# Cross Validation Classification Report
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
test_size = 0.33
seed = 7
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=test_size,
random_state=seed)
model = LogisticRegression()
model.fit(X_train, Y_train)
predicted = model.predict(X_test)
report = classification_report(Y_test, predicted)
print(report)

You can see good prediction and recall for the algorithm.

precision recall f1-score support
0.0 0.77 0.87 0.82 162
1.0 0.71 0.55 0.62 92

avg / total 0.75 0.76 0.75 254

##### 10.3 Regression Metrics
In this section will review 3 of the most common metrics for evaluating predictions on regression
machine learning problems:
 + Mean Absolute Error.
 + Mean Squared Error.
 + R2.

10.3.1 Mean Absolute Error
The Mean Absolute Error (or MAE) is the sum of the absolute differences between predictions
and actual values. It gives an idea of how wrong the predictions were. The measure gives an
idea of the magnitude of the error, but no idea of the direction (e.g. over or under predicting).
The example below demonstrates calculating mean absolute error on the Boston house price
dataset.

In [ ]:
# Cross Validation Regression MAE
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
filename = 'housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataframe = read_csv(filename, delim_whitespace=True, names=names)
array = dataframe.values
X = array[:,0:13]
Y = array[:,13]
kfold = KFold(n_splits=10, random_state=7)
model = LinearRegression()
scoring = 'neg_mean_absolute_error'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print("MAE: %.3f (%.3f)") % (results.mean(), results.std())

A value of 0 indicates no error or perfect predictions. Like logloss, this metric is inverted by
the cross val score() function.

MAE: -4.005 (2.084)

##### 10.3.2 Mean Squared Error
The Mean Squared Error (or MSE) is much like the mean absolute error in that it provides a
gross idea of the magnitude of error. Taking the square root of the mean squared error converts
the units back to the original units of the output variable and can be meaningful for description
and presentation. This is called the Root Mean Squared Error (or RMSE). The example below
provides a demonstration of calculating mean squared error.

In [ ]:
# Cross Validation Regression MSE
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
filename = 'housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataframe = read_csv(filename, delim_whitespace=True, names=names)
array = dataframe.values
X = array[:,0:13]
Y = array[:,13]
num_folds = 10
kfold = KFold(n_splits=10, random_state=7)
model = LinearRegression()
scoring = 'neg_mean_squared_error'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print("MSE: %.3f (%.3f)") % (results.mean(), results.std())

This metric too is inverted so that the results are increasing. Remember to take the absolute
value before taking the square root if you are interested in calculating the RMSE.

MSE: -34.705 (45.574)

##### 10.3.3 R2 Metric
The R2
(or R Squared) metric provides an indication of the goodness of fit of a set of predictions
to the actual values. In statistical literature this measure is called the coefficient of determination.
This is a value between 0 and 1 for no-fit and perfect fit respectively. The example below
provides a demonstration of calculating the mean R2

for a set of predictions.

In [ ]:
# Cross Validation Regression R^2
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
filename = 'housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataframe = read_csv(filename, delim_whitespace=True, names=names)
array = dataframe.values
X = array[:,0:13]
Y = array[:,13]

kfold = KFold(n_splits=10, random_state=7)
model = LinearRegression()
scoring = 'r2'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print("R^2: %.3f (%.3f)") % (results.mean(), results.std())

You can see the predictions have a poor fit to the actual values with a value closer to zero
and less than 0.5.

R^2: 0.203 (0.595)

10.4 Summary
In this chapter you discovered metrics that you can use to evaluate your machine learning
algorithms.
You learned about three classification metrics: Accuracy, Logarithmic Loss and Area Under
ROC Curve. You also learned about two convenience methods for classification prediction
results: the Confusion Matrix and the Classification Report. Finally, you also learned about
three metrics for regression problems: Mean Absolute Error, Mean Squared Error and R2

##### 10.4.1 Next
You now know how to evaluate the performance of machine learning algorithms using a variety
of different metrics and how to use those metrics to estimate the performance of algorithms on
new unseen data using resampling. In the next lesson you will start looking at machine learning
algorithms themselves, starting with classification techniques.

### Chapter 11: Spot-Check Classification Algorithms

Spot-checking is a way of discovering which algorithms perform well on your machine learning
problem. You cannot know which algorithms are best suited to your problem beforehand. You
must trial a number of methods and focus attention on those that prove themselves the most
promising. In this chapter you will discover six machine learning algorithms that you can use
when spot-checking your classification problem in Python with scikit-learn. After completing
this lesson you will know:
1. How to spot-check machine learning algorithms on a classification problem.
2. How to spot-check two linear classification algorithms.
3. How to spot-check four nonlinear classification algorithms.
Let’s get started.

##### 11.1 Algorithm Spot-Checking
You cannot know which algorithm will work best on your dataset beforehand. You must use
trial and error to discover a shortlist of algorithms that do well on your problem that you can
then double down on and tune further. I call this process spot-checking.
The question is not: What algorithm should I use on my dataset? Instead it is: What
algorithms should I spot-check on my dataset? You can guess at what algorithms might do
well on your dataset, and this can be a good starting point. I recommend trying a mixture of
algorithms and see what is good at picking out the structure in your data. Below are some
suggestions when spot-checking algorithms on your dataset:
 + Try a mixture of algorithm representations (e.g. instances and trees).
 + Try a mixture of learning algorithms (e.g. different algorithms for learning the same type
of representation).
 + Try a mixture of modeling types (e.g. linear and nonlinear functions or parametric and
nonparametric).
Let’s get specific. In the next section, we will look at algorithms that you can use to
spot-check on your next classification machine learning project in Python.

##### 11.2 Algorithms Overview
We are going to take a look at six classification algorithms that you can spot-check on your
dataset. Starting with two linear machine learning algorithms:
 + Logistic Regression.
 + Linear Discriminant Analysis.
Then looking at four nonlinear machine learning algorithms:
 + k-Nearest Neighbors.
 + Naive Bayes.
 + Classification and Regression Trees.
 + Support Vector Machines.
Each recipe is demonstrated on the Pima Indians onset of Diabetes dataset. A test harness
using 10-fold cross validation is used to demonstrate how to spot-check each machine learning
algorithm and mean accuracy measures are used to indicate algorithm performance. The recipes
assume that you know about each machine learning algorithm and how to use them. We will
not go into the API or parameterization of each algorithm.

##### 11.3 Linear Machine Learning Algorithms
This section demonstrates minimal recipes for how to use two linear machine learning algorithms:
logistic regression and linear discriminant analysis.

##### 11.3.1 Logistic Regression
Logistic regression assumes a Gaussian distribution for the numeric input variables and can
model binary classification problems. You can construct a logistic regression model using the
LogisticRegression class

In [ ]:
# Logistic Regression Classification
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
num_folds = 10
kfold = KFold(n_splits=10, random_state=7)
model = LogisticRegression()
results = cross_val_score(model, X, Y, cv=kfold)
print(results.mean())

Running the example prints the mean estimated accuracy.

0.76951469583

##### 11.3.2 Linear Discriminant Analysis
Linear Discriminant Analysis or LDA is a statistical technique for binary and multiclass
classification. It too assumes a Gaussian distribution for the numerical input variables. You can
construct an LDA model using the LinearDiscriminantAnalysis class.

In [ ]:
# LDA Classification
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
num_folds = 10
kfold = KFold(n_splits=10, random_state=7)
model = LinearDiscriminantAnalysis()
results = cross_val_score(model, X, Y, cv=kfold)
print(results.mean())

Running the example prints the mean estimated accuracy.

0.773462064252

##### 11.4 Nonlinear Machine Learning Algorithms
This section demonstrates minimal recipes for how to use 4 nonlinear machine learning algorithms.

##### 11.4.1 k-Nearest Neighbors
The k-Nearest Neighbors algorithm (or KNN) uses a distance metric to find the k most similar
instances in the training data for a new instance and takes the mean outcome of the neighbors
as the prediction. You can construct a KNN model using the KNeighborsClassifier class.

In [ ]:
# KNN Classification
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
num_folds = 10
kfold = KFold(n_splits=10, random_state=7)
model = KNeighborsClassifier()
results = cross_val_score(model, X, Y, cv=kfold)
print(results.mean())

Running the example prints the mean estimated accuracy.

0.726555023923

##### 11.4.2 Naive Bayes
Naive Bayes calculates the probability of each class and the conditional probability of each class
given each input value. These probabilities are estimated for new data and multiplied together,
assuming that they are all independent (a simple or naive assumption). When working with
real-valued data, a Gaussian distribution is assumed to easily estimate the probabilities for
input variables using the Gaussian Probability Density Function. You can construct a Naive
Bayes model using the GaussianNB class.

In [ ]:
# Gaussian Naive Bayes Classification
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import GaussianNB
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
kfold = KFold(n_splits=10, random_state=7)
model = GaussianNB()
results = cross_val_score(model, X, Y, cv=kfold)
print(results.mean())

Running the example prints the mean estimated accuracy.

0.75517771702

##### 11.4.3 Classification and Regression Trees
Classification and Regression Trees (CART or just decision trees) construct a binary tree from
the training data. Split points are chosen greedily by evaluating each attribute and each value
of each attribute in the training data in order to minimize a cost function (like the Gini index).
You can construct a CART model using the DecisionTreeClassifier class.

In [ ]:
# CART Classification
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
kfold = KFold(n_splits=10, random_state=7)
model = DecisionTreeClassifier()
results = cross_val_score(model, X, Y, cv=kfold)
print(results.mean())

Running the example prints the mean estimated accuracy.

0.692600820232

##### 11.4.4 Support Vector Machines
Support Vector Machines (or SVM) seek a line that best separates two classes. Those data
instances that are closest to the line that best separates the classes are called support vectors
and influence where the line is placed. SVM has been extended to support multiple classes.
Of particular importance is the use of different kernel functions via the kernel parameter. A powerful Radial Basis Function is used by default. You can construct an SVM model using the
SVC class.

In [ ]:
# SVM Classification
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
kfold = KFold(n_splits=10, random_state=7)
model = SVC()
results = cross_val_score(model, X, Y, cv=kfold)
print(results.mean())

Running the example prints the mean estimated accuracy.

0.651025290499

##### 11.5 Summary
In this chapter you discovered 6 machine learning algorithms that you can use to spot-check
on your classification problem in Python using scikit-learn. Specifically, you learned how to
spot-check two linear machine learning algorithms: Logistic Regression and Linear Discriminant
Analysis. You also learned how to spot-check four nonlinear algorithms: k-Nearest Neighbors,
Naive Bayes, Classification and Regression Trees and Support Vector Machines.

11.5.1 Next
In the next lesson you will discover how you can use spot-checking on regression machine learning
problems and practice with seven different regression algorithms.

### Chapter 12: Spot-Check Regression Algorithms

Spot-checking is a way of discovering which algorithms perform well on your machine learning
problem. You cannot know which algorithms are best suited to your problem beforehand. You
must trial a number of methods and focus attention on those that prove themselves the most
promising. In this chapter you will discover six machine learning algorithms that you can use
when spot-checking your regression problem in Python with scikit-learn. After completing this
lesson you will know:
1. How to spot-check machine learning algorithms on a regression problem.
2. How to spot-check four linear regression algorithms.
3. How to spot-check three nonlinear regression algorithms.
Let’s get started.

##### 12.1 Algorithms Overview
In this lesson we are going to take a look at seven regression algorithms that you can spot-check
on your dataset. Starting with four linear machine learning algorithms:
+ Linear Regression.
+ Ridge Regression.
+ LASSO Linear Regression.
+ Elastic Net Regression.
Then looking at three nonlinear machine learning algorithms:
+ k-Nearest Neighbors.
+ Classification and Regression Trees.
+ Support Vector Machines.

##### 12.2 Linear Machine Learning Algorithms
This section provides examples of how to use four different linear machine learning algorithms
for regression in Python with scikit-learn.

##### 12.2.1 Linear Regression
Linear regression assumes that the input variables have a Gaussian distribution. It is also
assumed that input variables are relevant to the output variable and that they are not highly
correlated with each other (a problem called collinearity). You can construct a linear regression
model using the LinearRegression class.

In [ ]:
# Linear Regression
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
filename = 'housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataframe = read_csv(filename, delim_whitespace=True, names=names)
array = dataframe.values
X = array[:,0:13]
Y = array[:,13]
kfold = KFold(n_splits=10, random_state=7)
model = LinearRegression()
scoring = 'neg_mean_squared_error'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print(results.mean())

Running the example provides a estimate of mean squared error.

-34.7052559445

##### 12.2.2 Ridge Regression
Ridge regression is an extension of linear regression where the loss function is modified to
minimize the complexity of the model measured as the sum squared value of the coefficient
values (also called the L2-norm). You can construct a ridge regression model by using the Ridge
class.

In [ ]:
# Ridge Regression
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
filename = 'housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataframe = read_csv(filename, delim_whitespace=True, names=names)
array = dataframe.values
X = array[:,0:13]
Y = array[:,13]
num_folds = 10
kfold = KFold(n_splits=10, random_state=7)
model = Ridge()
scoring = 'neg_mean_squared_error'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print(results.mean())

Running the example provides an estimate of the mean squared error.

-34.0782462093

12.2.3 LASSO Regression
The Least Absolute Shrinkage and Selection Operator (or LASSO for short) is a modification
of linear regression, like ridge regression, where the loss function is modified to minimize the
complexity of the model measured as the sum absolute value of the coefficient values (also called
the L1-norm). You can construct a LASSO model by using the Lasso class.

In [ ]:
# Lasso Regression
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Lasso
filename = 'housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataframe = read_csv(filename, delim_whitespace=True, names=names)
array = dataframe.values
X = array[:,0:13]
Y = array[:,13]
kfold = KFold(n_splits=10, random_state=7)
model = Lasso()
scoring = 'neg_mean_squared_error'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print(results.mean())

Running the example provides an estimate of the mean squared error.

-34.4640845883

##### 12.2.4 ElasticNet Regression
ElasticNet is a form of regularization regression that combines the properties of both Ridge
Regression and LASSO regression. It seeks to minimize the complexity of the regression model
(magnitude and number of regression coefficients) by penalizing the model using both the
L2-norm (sum squared coefficient values) and the L1-norm (sum absolute coefficient values).
You can construct an ElasticNet model using the ElasticNet class.

In [ ]:
# ElasticNet Regression
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import ElasticNet
filename = 'housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataframe = read_csv(filename, delim_whitespace=True, names=names)
array = dataframe.values
X = array[:,0:13]
Y = array[:,13]
kfold = KFold(n_splits=10, random_state=7)
model = ElasticNet()
scoring = 'neg_mean_squared_error'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print(results.mean())

Running the example provides an estimate of the mean squared error.

-31.1645737142

##### 12.3 Nonlinear Machine Learning Algorithms
This section provides examples of how to use three different nonlinear machine learning algorithms
for regression in Python with scikit-learn.

##### 12.3.1 K-Nearest Neighbors
The k-Nearest Neighbors algorithm (or KNN) locates the k most similar instances in the
training dataset for a new data instance. From the k neighbors, a mean or median output
variable is taken as the prediction. Of note is the distance metric used (the metric argument).
The Minkowski distance is used by default, which is a generalization of both the Euclidean
distance (used when all inputs have the same scale) and Manhattan distance (for when the
scales of the input variables differ). You can construct a KNN model for regression using the
KNeighborsRegressor class.

In [ ]:
# KNN Regression
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsRegressor
filename = 'housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataframe = read_csv(filename, delim_whitespace=True, names=names)
array = dataframe.values
X = array[:,0:13]
Y = array[:,13]
kfold = KFold(n_splits=10, random_state=7)
model = KNeighborsRegressor()
scoring = 'neg_mean_squared_error'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print(results.mean())

Running the example provides an estimate of the mean squared error.

-107.28683898

##### 12.3.2 Classification and Regression Trees

Decision trees or the Classification and Regression Trees (CART as they are known) use the train-
ing data to select the best points to split the data in order to minimize a cost metric. The default

cost metric for regression decision trees is the mean squared error, specified in the criterion
parameter. You can create a CART model for regression using the DecisionTreeRegressor
class.

In [ ]:
# Decision Tree Regression
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor
filename = 'housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataframe = read_csv(filename, delim_whitespace=True, names=names)
array = dataframe.values
X = array[:,0:13]
Y = array[:,13]
kfold = KFold(n_splits=10, random_state=7)
model = DecisionTreeRegressor()
scoring = 'neg_mean_squared_error'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print(results.mean())

Running the example provides an estimate of the mean squared error.

-35.4906027451

##### 12.3.3 Support Vector Machines
Support Vector Machines (SVM) were developed for binary classification. The technique has
been extended for the prediction real-valued problems called Support Vector Regression (SVR).
Like the classification example, SVR is built upon the LIBSVM library. You can create an SVM
model for regression using the SVR class.

In [ ]:
# SVM Regression
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVR
filename = 'housing.csv'
names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO',
'B', 'LSTAT', 'MEDV']
dataframe = read_csv(filename, delim_whitespace=True, names=names)
array = dataframe.values
X = array[:,0:13]
Y = array[:,13]
num_folds = 10
kfold = KFold(n_splits=10, random_state=7)
model = SVR()
scoring = 'neg_mean_squared_error'
results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
print(results.mean())

Running the example provides an estimate of the mean squared error.

-91.0478243332

##### 12.4 Summary
In this chapter you discovered how to spot-check machine learning algorithms for regression
problems in Python using scikit-learn. Specifically, you learned about four linear machine
learning algorithms: Linear Regression, Ridge Regression, LASSO Linear Regression and Elastic
Net Regression. You also learned about three nonlinear algorithms: k-Nearest Neighbors,
Classification and Regression Trees and Support Vector Machines.

##### 12.4.1 Next
Now that you know how to use classification and regression algorithms you need to know how
to compare the results of different algorithms to each other. In the next lesson you will discover
how to design simple experiments to directly compare machine learning algorithms to each other
on your dataset.

### Chapter 13: Compare Machine Learning Algorithms

It is important to compare the performance of multiple different machine learning algorithms
consistently. In this chapter you will discover how you can create a test harness to compare
multiple different machine learning algorithms in Python with scikit-learn. You can use this
test harness as a template on your own machine learning problems and add more and different
algorithms to compare. After completing this lesson you will know:
1. How to formulate an experiment to directly compare machine learning algorithms.
2. A reusable template for evaluating the performance of multiple algorithms on one dataset.
3. How to report and visualize the results when comparing algorithm performance.
Let’s get started.

##### 13.2 Compare Machine Learning Algorithms Consistently
The key to a fair comparison of machine learning algorithms is ensuring that each algorithm is
evaluated in the same way on the same data. You can achieve this by forcing each algorithm to be evaluated on a consistent test harness. In the example below six different classification
algorithms are compared on a single dataset:
+ Logistic Regression.
+ Linear Discriminant Analysis.
+ k-Nearest Neighbors.
+ Classification and Regression Trees.
+ Naive Bayes.
+ Support Vector Machines.
The dataset is the Pima Indians onset of diabetes problem. The problem has two classes and
eight numeric input variables of varying scales. The 10-fold cross validation procedure is used to
evaluate each algorithm, importantly configured with the same random seed to ensure that the
same splits to the training data are performed and that each algorithm is evaluated in precisely
the same way. Each algorithm is given a short name, useful for summarizing results afterward.

In [ ]:
# Compare Algorithms
from pandas import read_csv
from matplotlib import pyplot
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
# load dataset
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
# prepare models
models = []
models.append(('LR', LogisticRegression()))
models.append(('LDA', LinearDiscriminantAnalysis()))
models.append(('KNN', KNeighborsClassifier()))
models.append(('CART', DecisionTreeClassifier()))
models.append(('NB', GaussianNB()))
models.append(('SVM', SVC()))
# evaluate each model in turn
results = []
names = []
scoring = 'accuracy'
for name, model in models:
kfold = KFold(n_splits=10, random_state=7)
cv_results = cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
results.append(cv_results)
names.append(name)
msg = "%s: %f (%f)" % (name, cv_results.mean(), cv_results.std())
print(msg)
# boxplot algorithm comparison
fig = pyplot.figure()
fig.suptitle('Algorithm Comparison')
ax = fig.add_subplot(111)
pyplot.boxplot(results)
ax.set_xticklabels(names)
pyplot.show()

Running the example provides a list of each algorithm short name, the mean accuracy and
the standard deviation accuracy.

LR: 0.769515 (0.048411)
LDA: 0.773462 (0.051592)
KNN: 0.726555 (0.061821)
CART: 0.695232 (0.062517)
NB: 0.755178 (0.042766)
SVM: 0.651025 (0.072141)

##### 13.3 Summary
In this chapter you discovered how to evaluate multiple different machine learning algorithms
on a dataset in Python with scikit-learn. You learned how to both use the same test harness to
evaluate the algorithms and how to summarize the results both numerically and using a box
and whisker plot. You can use this recipe as a template for evaluating multiple algorithms on
your own problems.

13.3.1 Next
In this lesson you learned how to compare the performance of machine learning algorithms to
each other. But what if you need to prepare your data as part of the comparison process. In
the next lesson you will discover Pipelines in scikit-learn and how they overcome the common
problems of data leakage when comparing machine learning algorithms.

### Chapter 14: Automate Machine Learning Workflows with Pipelines

There are standard workflows in a machine learning project that can be automated. In Python
scikit-learn, Pipelines help to clearly define and automate these workflows. In this chapter you
will discover Pipelines in scikit-learn and how you can automate common machine learning
workflows. After completing this lesson you will know:
1. How to use pipelines to minimize data leakage.
2. How to construct a data preparation and modeling pipeline.
3. How to construct a feature extraction and modeling pipeline.
Let’s get started.

##### 14.1 Automating Machine Learning Workflows
There are standard workflows in applied machine learning. Standard because they overcome
common problems like data leakage in your test harness. Python scikit-learn provides a Pipeline
utility to help automate machine learning workflows. Pipelines work by allowing for a linear
sequence of data transforms to be chained together culminating in a modeling process that can
be evaluated.
The goal is to ensure that all of the steps in the pipeline are constrained to the data available
for the evaluation, such as the training dataset or each fold of the cross validation procedure.
You can learn more about Pipelines in scikit-learn by reading the Pipeline section1 of the user
guide. You can also review the API documentation for the Pipeline and FeatureUnion classes
and the pipeline module.

##### 14.2 Data Preparation and Modeling Pipeline
An easy trap to fall into in applied machine learning is leaking data from your training dataset
to your test dataset. To avoid this trap you need a robust test harness with strong separation of training and testing. This includes data preparation. Data preparation is one easy way to leak
knowledge of the whole training dataset to the algorithm. For example, preparing your data
using normalization or standardization on the entire training dataset before learning would not
be a valid test because the training dataset would have been influenced by the scale of the data
in the test set.
Pipelines help you prevent data leakage in your test harness by ensuring that data preparation
like standardization is constrained to each fold of your cross validation procedure. The example
below demonstrates this important data preparation and model evaluation workflow on the
Pima Indians onset of diabetes dataset. The pipeline is defined with two steps:
1. Standardize the data.
2. Learn a Linear Discriminant Analysis model.
The pipeline is then evaluated using 10-fold cross validation.

In [ ]:
# Create a pipeline that standardizes the data then creates a model
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
# load data
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
# create pipeline
estimators = []
estimators.append(('standardize', StandardScaler()))
estimators.append(('lda', LinearDiscriminantAnalysis()))
model = Pipeline(estimators)
# evaluate pipeline
kfold = KFold(n_splits=10, random_state=7)
results = cross_val_score(model, X, Y, cv=kfold)
print(results.mean())

Notice how we create a Python list of steps that are provided to the Pipeline for process
the data. Also notice how the Pipeline itself is treated like an estimator and is evaluated in its
entirety by the k-fold cross validation procedure. Running the example provides a summary of
accuracy of the setup on the dataset.

0.773462064252

##### 14.3 Feature Extraction and Modeling Pipeline
Feature extraction is another procedure that is susceptible to data leakage. Like data preparation,
feature extraction procedures must be restricted to the data in your training dataset. The
pipeline provides a handy tool called the FeatureUnion which allows the results of multiple
feature selection and extraction procedures to be combined into a larger dataset on which a
model can be trained. Importantly, all the feature extraction and the feature union occurs
within each fold of the cross validation procedure. The example below demonstrates the pipeline
defined with four steps:
1. Feature Extraction with Principal Component Analysis (3 features).
2. Feature Extraction with Statistical Selection (6 features).
3. Feature Union.
4. Learn a Logistic Regression Model.
The pipeline is then evaluated using 10-fold cross validation.

In [ ]:
# Create a pipeline that extracts features from the data then creates a model
from pandas import read_csv
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest
# load data
filename = 'pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = read_csv(filename, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
# create feature union
features = []
features.append(('pca', PCA(n_components=3)))
features.append(('select_best', SelectKBest(k=6)))
feature_union = FeatureUnion(features)
# create pipeline
estimators = []
estimators.append(('feature_union', feature_union))
estimators.append(('logistic', LogisticRegression()))
model = Pipeline(estimators)
# evaluate pipeline
kfold = KFold(n_splits=10, random_state=7)
results = cross_val_score(model, X, Y, cv=kfold)
print(results.mean())

Notice how the FeatureUnion is it’s own Pipeline that in turn is a single step in the final
Pipeline used to feed Logistic Regression. This might get you thinking about how you can start
embedding pipelines within pipelines. Running the example provides a summary of accuracy of
the setup on the dataset.

0.776042378674

##### 14.4 Summary
In this chapter you discovered the difficulties of data leakage in applied machine learning. You
discovered the Pipeline utilities in Python scikit-learn and how they can be used to automate
standard applied machine learning workflows. You learned how to use Pipelines in two important
use cases:
+ Data preparation and modeling constrained to each fold of the cross validation procedure.
+ Feature extraction and feature union constrained to each fold of the cross validation
procedure.
14.4.1 Next
This completes the lessons on how to evaluate machine learning algorithms. In the next lesson
you will take your first look at how to improve algorithm performance on your problems by
using ensemble methods.